# Allegiant Air — Configurable PII Masking in Nested JSON (POC)

Surgically masks PII **inside** nested Google-Analytics-style JSON while **preserving the JSON
structure and every non-PII value** — the ask in Kartik Oswal's 2026-07-09 email.

**Runtime is native SQL (`regexp_replace`).** The masking config (`pii_policies`) is
*compiled* at deploy time into a `regexp_replace` chain that's baked into one **SQL** mask function per
governed-tag value. That keeps it **config-driven** *and* fast: the read path stays in **Photon**. Enforcement is tag-driven **ABAC**: one schema-level policy per tag value auto-masks
every column carrying that tag (column-agnostic; covers current + future tables).

| Object | Role |
|---|---|
| `mask_expr(col, cfg)` (deploy-time) | compiles a config into a native `regexp_replace` expression |
| `mask_pii_<value>(payload)` (SQL UDF) | native mask fn per tag value; body = compiled `regexp_replace` chain |
| governed tag `pii = <value>` | marks which columns hold which kind of PII |
| ABAC policy `pii_json_mask_<value>` | schema-level; auto-masks every column tagged with that value |

**Strategies:** `REDACT` (`***MASKED***`), `NULLIFY`, `PARTIAL` (first char + `***`). *(HASH was dropped —
it can't be done in `regexp_replace`; if you need deterministic hashing, do it in a batch/materialize
step, not at read time.)* **Config fields:** `keys` (string value of a key at any depth), `url_params`
(query-param value inside a string), `regex` (pattern inside a string).

## 0. Configuration — everything below is driven by these widgets

In [0]:
import json

# Clear any stale widgets (e.g. renamed/removed ones from an earlier notebook version) so the set defined
# below is authoritative — keeps the notebook idempotent. NOTE: this resets widget values to the defaults,
# so set any custom values AFTER this cell has run.
dbutils.widgets.removeAll()

dbutils.widgets.text("catalog",           "dkushari_uc",   "1 Catalog")
dbutils.widgets.text("schema",            "allegiant_air", "2 Schema")
dbutils.widgets.text("table",             "ga_app_hits",   "3 Table")
dbutils.widgets.text("num_records",       "10000",         "4 Demo record count")
dbutils.widgets.text("tag_key",           "pii_aa",        "5 Governed tag key")
dbutils.widgets.text("full_access_group", "",              "6 Account group that sees RAW (blank=mask everyone)")
# Map: governed-tag value -> masking config. ONE native SQL mask fn + ABAC policy is created per entry.
# The default is loaded from config/pii_policies.json (the repo file is the single source of truth); an
# inline fallback keeps the notebook runnable standalone (outside the repo). Edit the file OR the widget.
import os
_INLINE_POLICIES = {
    "name":  {"strategy": "REDACT", "keys": ["name"],
              "url_params": ["firstName", "lastName", "fname", "lname", "first_name", "last_name", "email"], "regex": []},
    "email": {"strategy": "REDACT", "keys": ["email"], "url_params": [], "regex": []},
}
def _load_pii_policies():
    try:
        nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        path = "/Workspace" + os.path.dirname(os.path.dirname(nb)) + "/config/pii_policies.json"  # ../config/
        with open(path) as f:
            print(f"pii_policies default loaded from {path}")
            return json.load(f)
    except Exception as e:
        print(f"pii_policies: using inline default (config/pii_policies.json not read: {str(e)[:80]})")
        return _INLINE_POLICIES
dbutils.widgets.text("pii_policies", json.dumps(_load_pii_policies()), "7 PII policies (tag_value -> config)")
# Map: column -> governed-tag value. Which columns to tag in the demo table.
dbutils.widgets.text("tag_columns", json.dumps({"hit_payload": "name", "user_payload": "email"}),
                     "8 Columns to tag (column -> tag_value)")

catalog           = dbutils.widgets.get("catalog")
schema            = dbutils.widgets.get("schema")
table             = dbutils.widgets.get("table")
num_records       = int(dbutils.widgets.get("num_records"))
tag_key           = dbutils.widgets.get("tag_key")
full_access_group = dbutils.widgets.get("full_access_group").strip()
pii_policies      = json.loads(dbutils.widgets.get("pii_policies"))   # validate early
tag_columns       = json.loads(dbutils.widgets.get("tag_columns"))
fqtn              = f"{catalog}.{schema}.{table}"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"DROP SCHEMA IF EXISTS {catalog}.{schema} CASCADE")  # clean-slate reset so every run is fully idempotent (cascade drops tables, UDFs, and ABAC policies in the schema)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema} COMMENT 'Allegiant Air POC — configurable PII masking in nested JSON'")

print("Target table       :", fqtn)
print(f"Governed tag key   : {tag_key}")
print("Full-access group  :", full_access_group or "(none — everyone masked)")
print("PII policies       :", json.dumps(pii_policies, indent=2))
print("Column -> tag value:", json.dumps(tag_columns, indent=2))
print("\nNOTE: §4 self-provisions the governed tag (create it / add values / no-op). If a governed tag")
print(f"'{tag_key}' already exists (fixed allowed-value list), each value must be in that list.")

pii_policies default loaded from /Workspace/Users/dipankar.kushari@databricks.com/allegiantair/allegiant-json-pii-masking/config/pii_policies.json
Target table       : dkushari_uc.allegiant_air.ga_app_hits
Governed tag key   : pii_aa
Full-access group  : (none — everyone masked)
PII policies       : {
  "name": {
    "strategy": "REDACT",
    "keys": [
      "name"
    ],
    "paths": [],
    "url_params": [
      "firstName",
      "lastName",
      "fname",
      "lname",
      "first_name",
      "last_name",
      "email"
    ],
    "regex": []
  },
  "email": {
    "strategy": "REDACT",
    "keys": [
      "email"
    ],
    "paths": [],
    "url_params": [],
    "regex": []
  }
}
Column -> tag value: {
  "hit_payload": "name",
  "user_payload": "email"
}

NOTE: §4 self-provisions the governed tag (create it / add values / no-op). If a governed tag
'pii_aa' already exists (fixed allowed-value list), each value must be in that list.


## 1. Masking engine — native SQL, compiled from config (`mask_expr`)
`mask_expr(col, cfg)` returns a native **`regexp_replace`** expression that masks a JSON **string**
column per a config. It's used both to preview (§3) and to build the per-tag SQL mask functions (§5).
The generated SQL stays in Photon on the read path.

- `keys` → masks the string value of `"<key>": "..."` at any depth (quote-anchored, so `name` never
  matches `appName`).
- `url_params` → masks the value after `?param=` / `&param=` inside any string (e.g. a URL).
- `regex` → replaces matches of a user pattern inside any string.
- Strategy: `REDACT`→`***MASKED***`, `NULLIFY`→removed, `PARTIAL`→first char + `***`.

In [0]:
def mask_expr(col, cfg):
    """Return a native-SQL regexp_replace expression masking `col` (a STRING JSON) per `cfg`.
    Supports keys / url_params / regex; strategies REDACT | NULLIFY | PARTIAL (no HASH at read time).
    Patterns use ' *' (not \\s) to avoid backslash-escaping issues across the SQL/regex layers."""
    strat = (cfg.get("strategy") or "REDACT").upper()
    tok = "***" if strat == "PARTIAL" else "***MASKED***"
    e = col
    for k in (cfg.get("keys") or []):
        if strat == "NULLIFY":
            e = f"""regexp_replace({e}, '("{k}" *: *)"[^"]*"', '$1null')"""
        elif strat == "PARTIAL":
            e = f"""regexp_replace({e}, '("{k}" *: *")([^"])[^"]*"', '$1$2***"')"""
        else:  # REDACT
            e = f"""regexp_replace({e}, '("{k}" *: *)"[^"]*"', '$1"{tok}"')"""
    ups = cfg.get("url_params") or []
    if ups:
        alt = "|".join(ups)
        repl = "$1" if strat == "NULLIFY" else f"$1{tok}"
        e = f"""regexp_replace({e}, '([?&]({alt})=)[^&#"]*', '{repl}')"""
    for rx in (cfg.get("regex") or []):
        repl = "" if strat == "NULLIFY" else tok
        e = f"""regexp_replace({e}, '{rx}', '{repl}')"""
    return e

print("Example compiled expression for the 'name' policy:\n")
print(mask_expr("hit_payload", pii_policies.get("name", {})))

Example compiled expression for the 'name' policy:

regexp_replace(regexp_replace(hit_payload, '("name" *: *)"[^"]*"', '$1"***MASKED***"'), '([?&](firstName|lastName|fname|lname|first_name|last_name|email)=)[^&#"]*', '$1***MASKED***')


## 2. Demo table — TWO nested-JSON columns, each with a different kind of PII
- `hit_payload` — `appInfo.name` + a `documentLocation` `manage-travel` URL carrying
  `firstName`/`lastName`/`email`. Tagged `pii=name`; the name policy REDACTs the `name` key AND all three
  URL params (incl. the email embedded in the URL).
- `user_payload` — `contact.email` (+ `phone` and loyalty left as non-PII). Tagged `pii=email`; the email
  policy REDACTs the `email` key. (Same email → REDACT in the URL and as a field.)

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {fqtn}
COMMENT 'GA-style abc2Go events. hit_payload: name PII + URL PII (tag pii=name). user_payload: email PII (tag pii=email). Multi-column ABAC demo.'
AS
SELECT id AS hit_id,
  to_json(named_struct(
    'appInfo', named_struct('appId','com.lixar.abc','appName','abc2Go','name',concat(fn,' ',ln),
                            'version',ver,'screenName',scr),
    'customDimensions', array(named_struct('index',6,'value',element_at(array('GUEST','MEMBER'),cast(rand()*2 AS INT)+1))),
    'documentLocation', concat('https://www.allegiant.com/manage-travel?firstName=',fn,'&lastName=',ln,'&email=',lower(fn),'.',lower(ln),'@example.com&pnr=',pnr),
    'hitNumber', hn, 'type','APPVIEW'
  )) AS hit_payload,
  to_json(named_struct(
    'contact', named_struct('email', concat(lower(fn),'.',lower(ln),'@example.com'),
                            'phone', concat('+1', cast(cast(rand()*9000000000+1000000000 AS BIGINT) AS STRING))),
    'loyalty', named_struct('tier', element_at(array('GOLD','SILVER','BRONZE'),cast(rand()*3 AS INT)+1),
                            'points', cast(rand()*100000 AS INT))
  )) AS user_payload
FROM (
  SELECT id,
    element_at(array('James','Mary','John','Patricia','Robert','Jennifer','Michael','Linda','William','Elizabeth','David','Barbara','Richard','Susan','Joseph','Jessica','Thomas','Sarah','Charles','Karen'), cast(rand()*20 AS INT)+1) AS fn,
    element_at(array('Smith','Johnson','Williams','Brown','Jones','Garcia','Miller','Davis','Rodriguez','Martinez','Hernandez','Lopez','Gonzalez','Wilson','Anderson','Thomas','Taylor','Moore','Jackson','Martin'), cast(rand()*20 AS INT)+1) AS ln,
    element_at(array('5.2.0','5.3.1','6.0.0','5.9.2'), cast(rand()*4 AS INT)+1) AS ver,
    element_at(array('Checkin Flow','Booking Flow','Seat Map','Payment','Confirmation','Home','Manage Travel'), cast(rand()*7 AS INT)+1) AS scr,
    upper(substr(md5(cast(rand() AS STRING)),1,6)) AS pnr, cast(rand()*6+1 AS INT) AS hn
  FROM range({num_records})
)
""")
print(f"Created {fqtn} with {num_records} rows (columns: hit_payload, user_payload)")

Created dkushari_uc.allegiant_air.ga_app_hits with 10000 rows (columns: hit_payload, user_payload)


### 2a. Raw data — PII visible before masking

In [0]:
display(spark.sql(f"""
SELECT hit_id,
       get_json_object(hit_payload,  '$.appInfo.name')     AS name_pii,
       get_json_object(hit_payload,  '$.documentLocation') AS url_pii,
       get_json_object(user_payload, '$.contact.email')    AS email_pii,
       get_json_object(user_payload, '$.contact.phone')    AS phone_nonpii,
       get_json_object(user_payload, '$.loyalty.tier')     AS tier_nonpii
FROM {fqtn} ORDER BY hit_id LIMIT 10
"""))

hit_id,name_pii,url_pii,email_pii,phone_nonpii,tier_nonpii
0,Robert Miller,https://www.allegiant.com/manage-travel?firstName=Robert&lastName=Miller&email=robert.miller@example.com&pnr=08F0E4,robert.miller@example.com,+16897280688,BRONZE
1,Susan Jones,https://www.allegiant.com/manage-travel?firstName=Susan&lastName=Jones&email=susan.jones@example.com&pnr=A7BCF4,susan.jones@example.com,+14012531129,BRONZE
2,John Anderson,https://www.allegiant.com/manage-travel?firstName=John&lastName=Anderson&email=john.anderson@example.com&pnr=43D8A3,john.anderson@example.com,+14006746014,BRONZE
3,Barbara Johnson,https://www.allegiant.com/manage-travel?firstName=Barbara&lastName=Johnson&email=barbara.johnson@example.com&pnr=9BFAC7,barbara.johnson@example.com,+13531307724,SILVER
4,Linda Gonzalez,https://www.allegiant.com/manage-travel?firstName=Linda&lastName=Gonzalez&email=linda.gonzalez@example.com&pnr=EBB7C4,linda.gonzalez@example.com,+19140852431,GOLD
5,James Gonzalez,https://www.allegiant.com/manage-travel?firstName=James&lastName=Gonzalez&email=james.gonzalez@example.com&pnr=0C9DBA,james.gonzalez@example.com,+12204459861,BRONZE
6,Barbara Brown,https://www.allegiant.com/manage-travel?firstName=Barbara&lastName=Brown&email=barbara.brown@example.com&pnr=46BB27,barbara.brown@example.com,+13717907239,BRONZE
7,James Johnson,https://www.allegiant.com/manage-travel?firstName=James&lastName=Johnson&email=james.johnson@example.com&pnr=B76A83,james.johnson@example.com,+17217334726,SILVER
8,William Wilson,https://www.allegiant.com/manage-travel?firstName=William&lastName=Wilson&email=william.wilson@example.com&pnr=C02493,william.wilson@example.com,+15199879789,BRONZE
9,Charles Jones,https://www.allegiant.com/manage-travel?firstName=Charles&lastName=Jones&email=charles.jones@example.com&pnr=FE2C49,charles.jones@example.com,+13422557320,GOLD


## 3. Test each policy's config directly (before / after) — native SQL
Applies the compiled `regexp_replace` expression so you can see the effect before enforcing.

In [0]:
name_expr_hit  = mask_expr("hit_payload",  pii_policies.get("name",  {"strategy": "REDACT", "keys": ["name"]}))
email_expr_usr = mask_expr("user_payload", pii_policies.get("email", {"strategy": "REDACT", "keys": ["email"]}))
display(spark.sql(f"""
SELECT
  get_json_object(hit_payload,'$.appInfo.name')                       AS name_raw,
  get_json_object({name_expr_hit},  '$.appInfo.name')                 AS name_masked,
  get_json_object(user_payload,'$.contact.email')                     AS email_raw,
  get_json_object({email_expr_usr}, '$.contact.email')                AS email_masked
FROM {fqtn} ORDER BY hit_id LIMIT 10
"""))

name_raw,name_masked,email_raw,email_masked
Robert Miller,***MASKED***,robert.miller@example.com,***MASKED***
Susan Jones,***MASKED***,susan.jones@example.com,***MASKED***
John Anderson,***MASKED***,john.anderson@example.com,***MASKED***
Barbara Johnson,***MASKED***,barbara.johnson@example.com,***MASKED***
Linda Gonzalez,***MASKED***,linda.gonzalez@example.com,***MASKED***
James Gonzalez,***MASKED***,james.gonzalez@example.com,***MASKED***
Barbara Brown,***MASKED***,barbara.brown@example.com,***MASKED***
James Johnson,***MASKED***,james.johnson@example.com,***MASKED***
William Wilson,***MASKED***,william.wilson@example.com,***MASKED***
Charles Jones,***MASKED***,charles.jones@example.com,***MASKED***


### 3b. Full document — original vs masked (structure preserved)
Prints the **entire** JSON before and after masking for one row, so you can see that *only* the PII
fields change and the whole structure + every non-PII value is preserved.

In [0]:
r = spark.sql(f"""
  SELECT hit_payload  AS hit_orig,  {name_expr_hit}  AS hit_masked,
         user_payload AS usr_orig,  {email_expr_usr} AS usr_masked
  FROM {fqtn} ORDER BY hit_id LIMIT 1
""").collect()[0]

def _show(title, original, masked):
    print("=" * 92); print(title); print("=" * 92)
    print("----- ORIGINAL -----"); print(json.dumps(json.loads(original), indent=2, ensure_ascii=False))
    print("----- MASKED   -----"); print(json.dumps(json.loads(masked),   indent=2, ensure_ascii=False))
    print()

_show("hit_payload  — pii=name: appInfo.name + URL firstName/lastName/email masked; all else preserved",
      r["hit_orig"], r["hit_masked"])
_show("user_payload — pii=email: contact.email masked; phone / loyalty preserved",
      r["usr_orig"], r["usr_masked"])

hit_payload  — pii=name: appInfo.name + URL firstName/lastName/email masked; all else preserved
----- ORIGINAL -----
{
  "appInfo": {
    "appId": "com.lixar.abc",
    "appName": "abc2Go",
    "name": "Robert Miller",
    "version": "5.9.2",
    "screenName": "Booking Flow"
  },
  "customDimensions": [
    {
      "index": 6,
      "value": "MEMBER"
    }
  ],
  "documentLocation": "https://www.allegiant.com/manage-travel?firstName=Robert&lastName=Miller&email=robert.miller@example.com&pnr=08F0E4",
  "hitNumber": 4,
  "type": "APPVIEW"
}
----- MASKED   -----
{
  "appInfo": {
    "appId": "com.lixar.abc",
    "appName": "abc2Go",
    "name": "***MASKED***",
    "version": "5.9.2",
    "screenName": "Booking Flow"
  },
  "customDimensions": [
    {
      "index": 6,
      "value": "MEMBER"
    }
  ],
  "documentLocation": "https://www.allegiant.com/manage-travel?firstName=***MASKED***&lastName=***MASKED***&email=***MASKED***&pnr=08F0E4",
  "hitNumber": 4,
  "type": "APPVIEW"
}

user_payl

## 4. Governed tag — self-provision, then tag the columns
Ensures the governed tag key exists and allows **every** value used in `pii_policies` (create it /
add missing values / no-op). Creating/altering a governed tag needs the **account-level `CREATE`**
privilege (account/workspace admin); without it the cell prints the DDL and continues.

In [0]:
# 4a. Ensure the governed tag exists and allows every value in pii_policies.
required_values = sorted(set(pii_policies.keys()) | set(tag_columns.values()))

def _governed_tag_values(key):
    """Allowed values of a governed tag, or None if it doesn't exist. Uses SHOW GOVERNED TAGS so a
    missing tag is simply absent (no failed DESCRIBE statement / no red error in query history)."""
    for r in spark.sql("SHOW GOVERNED TAGS").collect():
        if r["Tag Key"] == key:
            raw = r["Values"]                      # JSON array string, e.g. '["name","email"]' or '[]'
            try:
                parsed = list(json.loads(raw)) if raw else []
            except Exception:
                parsed = str(raw).strip("[]").split(",")
            # normalize: strip whitespace and any stray quotes so comparisons/ALTER never double-quote
            return [str(v).strip().strip('"').strip("'") for v in parsed if str(v).strip()]
    return None  # tag key does not exist

try:
    values = _governed_tag_values(tag_key)
    if values is None:                                   # neither key nor values exist -> create
        vlist = ", ".join(f"'{v}'" for v in required_values)
        spark.sql(f"CREATE GOVERNED TAG {tag_key} DESCRIPTION 'PII classification for ABAC column masking' VALUES ({vlist})")
        print(f"Created governed tag '{tag_key}' with values {required_values}.")
    else:
        missing = [v for v in required_values if v not in values]
        if missing:                                      # key exists, some values missing -> add them
            merged = values + missing                    # ALTER ... SET VALUES is declarative (replaces set)
            vlist = ", ".join(f"'{v}'" for v in merged)
            spark.sql(f"ALTER GOVERNED TAG {tag_key} SET VALUES ({vlist})")
            print(f"Added {missing} to governed tag '{tag_key}'. Allowed values now: {merged}")
        else:
            print(f"Governed tag '{tag_key}' already allows {required_values} — no change.")
except Exception as e:
    print("WARNING: could not create/alter the governed tag automatically — needs account-level CREATE "
          "(account/workspace admin). Ask an admin to run e.g.:\n"
          f"    CREATE GOVERNED TAG {tag_key} DESCRIPTION 'PII classification' VALUES ({', '.join(repr(v) for v in required_values)});\n"
          f"Details: {str(e)[:300]}")

Governed tag 'pii_aa' already allows ['email', 'name'] — no change.


In [0]:
# 4b. Tag each column with its governed-tag value.
for col, val in tag_columns.items():
    spark.sql(f"ALTER TABLE {fqtn} ALTER COLUMN {col} SET TAGS ('{tag_key}' = '{val}')")
    print(f"Tagged {fqtn}.{col} with {tag_key}={val}")
display(spark.sql(f"""
SELECT column_name, tag_name, tag_value
FROM {catalog}.information_schema.column_tags
WHERE schema_name='{schema}' AND table_name='{table}' ORDER BY column_name
"""))

Tagged dkushari_uc.allegiant_air.ga_app_hits.hit_payload with pii_aa=name
Tagged dkushari_uc.allegiant_air.ga_app_hits.user_payload with pii_aa=email


column_name,tag_name,tag_value
hit_payload,pii_aa,name
user_payload,pii_aa,email


## 5. Native mask functions + ABAC policies — one per PII type
For each `pii_policies` entry we compile the config into a **native SQL** mask function
`mask_pii_<value>(payload)` (body = `regexp_replace` chain), then create a schema-level ABAC
policy that applies it to every column tagged `{tag_key}=<value>`. No `USING COLUMNS` — the config is
baked into the function. Retarget by editing `pii_policies` and re-running (CREATE OR REPLACE).

In [0]:
except_clause = f"EXCEPT `{full_access_group}`" if full_access_group else ""
for val, cfg in pii_policies.items():
    body = mask_expr("payload", cfg)
    spark.sql(f"""
    CREATE OR REPLACE FUNCTION {catalog}.{schema}.mask_pii_{val}(payload STRING)
    RETURNS STRING
    COMMENT 'Native regexp_replace column mask for {tag_key}={val} (compiled from pii_policies).'
    RETURN {body}
    """)
    spark.sql(f"""
    CREATE OR REPLACE POLICY pii_json_mask_{val}
    ON SCHEMA {catalog}.{schema}
    COMMENT 'ABAC: auto-mask columns tagged {tag_key}={val} via native mask_pii_{val}.'
    COLUMN MASK {catalog}.{schema}.mask_pii_{val}
    TO `account users` {except_clause}
    FOR TABLES
    MATCH COLUMNS has_tag_value('{tag_key}', '{val}') AS c
    ON COLUMN c
    """)
    print(f"Created native mask fn + ABAC policy for {tag_key}={val}")
display(spark.sql(f"SHOW EFFECTIVE POLICIES ON TABLE {fqtn}"))

Created native mask fn + ABAC policy for pii_aa=name
Created native mask fn + ABAC policy for pii_aa=email


Policy Name,Policy Type,Catalog,Schema,Table,Comment
pii_json_mask_email,COLUMN_MASK,dkushari_uc,allegiant_air,null,ABAC: auto-mask columns tagged pii_aa=email via native mask_pii_email.
pii_json_mask_name,COLUMN_MASK,dkushari_uc,allegiant_air,null,ABAC: auto-mask columns tagged pii_aa=name via native mask_pii_name.


### 5a. What users see now — each tagged column masked per its policy, non-PII intact

In [0]:
display(spark.sql(f"""
SELECT hit_id,
       get_json_object(hit_payload,  '$.appInfo.name')     AS name,
       get_json_object(hit_payload,  '$.appInfo.appName')  AS app_name,
       get_json_object(hit_payload,  '$.documentLocation') AS url,
       get_json_object(user_payload, '$.contact.email')    AS email,
       get_json_object(user_payload, '$.contact.phone')    AS phone,
       get_json_object(user_payload, '$.loyalty.tier')     AS tier
FROM {fqtn} ORDER BY hit_id LIMIT 10
"""))

hit_id,name,app_name,url,email,phone,tier
0,***MASKED***,abc2Go,https://www.allegiant.com/manage-travel?firstName=***MASKED***&lastName=***MASKED***&email=***MASKED***&pnr=08F0E4,***MASKED***,+16897280688,BRONZE
1,***MASKED***,abc2Go,https://www.allegiant.com/manage-travel?firstName=***MASKED***&lastName=***MASKED***&email=***MASKED***&pnr=A7BCF4,***MASKED***,+14012531129,BRONZE
2,***MASKED***,abc2Go,https://www.allegiant.com/manage-travel?firstName=***MASKED***&lastName=***MASKED***&email=***MASKED***&pnr=43D8A3,***MASKED***,+14006746014,BRONZE
3,***MASKED***,abc2Go,https://www.allegiant.com/manage-travel?firstName=***MASKED***&lastName=***MASKED***&email=***MASKED***&pnr=9BFAC7,***MASKED***,+13531307724,SILVER
4,***MASKED***,abc2Go,https://www.allegiant.com/manage-travel?firstName=***MASKED***&lastName=***MASKED***&email=***MASKED***&pnr=EBB7C4,***MASKED***,+19140852431,GOLD
5,***MASKED***,abc2Go,https://www.allegiant.com/manage-travel?firstName=***MASKED***&lastName=***MASKED***&email=***MASKED***&pnr=0C9DBA,***MASKED***,+12204459861,BRONZE
6,***MASKED***,abc2Go,https://www.allegiant.com/manage-travel?firstName=***MASKED***&lastName=***MASKED***&email=***MASKED***&pnr=46BB27,***MASKED***,+13717907239,BRONZE
7,***MASKED***,abc2Go,https://www.allegiant.com/manage-travel?firstName=***MASKED***&lastName=***MASKED***&email=***MASKED***&pnr=B76A83,***MASKED***,+17217334726,SILVER
8,***MASKED***,abc2Go,https://www.allegiant.com/manage-travel?firstName=***MASKED***&lastName=***MASKED***&email=***MASKED***&pnr=C02493,***MASKED***,+15199879789,BRONZE
9,***MASKED***,abc2Go,https://www.allegiant.com/manage-travel?firstName=***MASKED***&lastName=***MASKED***&email=***MASKED***&pnr=FE2C49,***MASKED***,+13422557320,GOLD


### 5a-2. Full masked document — what users actually see (post-policy)
The columns now read through the ABAC policy, so this is the **enforced** output. Pretty-prints the
entire masked JSON for one row — same "only PII changed, structure preserved" view as §3b, but this is
what any (non-full-access) user gets at query time.

In [0]:
mr = spark.sql(f"SELECT hit_payload, user_payload FROM {fqtn} ORDER BY hit_id LIMIT 1").collect()[0]

def _show_masked(title, doc):
    print("=" * 92); print(title); print("=" * 92)
    print(json.dumps(json.loads(doc), indent=2, ensure_ascii=False)); print()

_show_masked("hit_payload  — as users see it (ABAC-masked): name + URL firstName/lastName/email masked", mr["hit_payload"])
_show_masked("user_payload — as users see it (ABAC-masked): contact.email masked; phone / loyalty intact", mr["user_payload"])

hit_payload  — as users see it (ABAC-masked): name + URL firstName/lastName/email masked
{
  "appInfo": {
    "appId": "com.lixar.abc",
    "appName": "abc2Go",
    "name": "***MASKED***",
    "version": "5.9.2",
    "screenName": "Booking Flow"
  },
  "customDimensions": [
    {
      "index": 6,
      "value": "MEMBER"
    }
  ],
  "documentLocation": "https://www.allegiant.com/manage-travel?firstName=***MASKED***&lastName=***MASKED***&email=***MASKED***&pnr=08F0E4",
  "hitNumber": 4,
  "type": "APPVIEW"
}

user_payload — as users see it (ABAC-masked): contact.email masked; phone / loyalty intact
{
  "contact": {
    "email": "***MASKED***",
    "phone": "+16897280688"
  },
  "loyalty": {
    "tier": "BRONZE",
    "points": 14786
  }
}



### 5b. Validate — every tagged column masked, non-PII preserved

In [0]:
display(spark.sql(f"""
SELECT count(*) AS total,
  sum(CASE WHEN get_json_object(hit_payload,'$.appInfo.name')='***MASKED***' THEN 1 ELSE 0 END)                                        AS name_masked,
  sum(CASE WHEN get_json_object(hit_payload,'$.documentLocation') LIKE '%firstName=***MASKED***%lastName=***MASKED***%email=***MASKED***%' THEN 1 ELSE 0 END) AS url_masked,
  sum(CASE WHEN get_json_object(user_payload,'$.contact.email')='***MASKED***' THEN 1 ELSE 0 END)                                      AS email_masked,
  sum(CASE WHEN get_json_object(user_payload,'$.contact.phone') LIKE '+1%' THEN 1 ELSE 0 END)                                          AS phone_preserved,
  sum(CASE WHEN get_json_object(hit_payload,'$.appInfo.appName')='abc2Go' THEN 1 ELSE 0 END)                                           AS appname_preserved
FROM {fqtn}
"""))

total,name_masked,url_masked,email_masked,phone_preserved,appname_preserved
10000,10000,10000,10000,10000,10000


## 6. Operate — scale, retarget, grant raw, remove, materialize (performance)
- **Add a PII type:** add an entry to `pii_policies` and re-run §5 — one more native fn + policy; tag any
  column `{tag_key}=<value>` to mask it.
- **Add a column / table:** just tag its column — the existing policy masks it automatically (column-agnostic).
- **Retarget:** edit a config in `pii_policies` and re-run §5.
- **Grant raw access:** set `full_access_group` to an **account** group and re-run §5.
- **Remove:** `DROP POLICY pii_json_mask_<value> ON SCHEMA {catalog}.{schema}` / `... UNSET TAGS ('{tag_key}')`.
- **Performance (recommended for "everyone masked"):** since nobody sees raw, skip per-query masking and
  **materialize** a masked table once (below) — reads then have zero masking cost.

In [0]:
# Materialize a masked copy (native, one-time) — fastest for the "everyone masked" posture. Uncomment to run.
# spark.sql(f"""
# CREATE OR REPLACE TABLE {fqtn}_masked AS
# SELECT hit_id, {mask_expr('hit_payload', pii_policies['name'])} AS hit_payload,
#                {mask_expr('user_payload', pii_policies['email'])} AS user_payload
# FROM {fqtn}
# """)
# print(f"Materialized {fqtn}_masked — SELECTs read plain columns, no per-query masking.")

## Appendix
- **VARIANT columns:** if the JSON is stored as `VARIANT`, wrap the mask fn body as
  `parse_json(regexp_replace(to_json(payload), …))` and return `VARIANT`. VARIANT speeds reads/analytics;
  masking still uses `regexp_replace` on the text form.
- **AI discovery:** `02_pii_path_discovery.py` proposes which keys/paths are PII (classify → review → tag).